In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

data = pd.read_excel('mpipe_video_latency_results.xlsx')

print(data.head().to_string())
df = pd.DataFrame(data)

                      video video_resolution  threshold  video_fps  frame_skip  latency_ms  processed_frames  total_frames
0  ejercicio02 - frente.mp4          144x256        0.1      29.96           5  105.443239                 1           513
1  ejercicio02 - frente.mp4          144x256        0.1      29.96           5   85.982084                 2           513
2  ejercicio02 - frente.mp4          144x256        0.1      29.96           5  120.065212                 3           513
3  ejercicio02 - frente.mp4          144x256        0.1      29.96           5   84.601879                 4           513
4  ejercicio02 - frente.mp4          144x256        0.1      29.96           5   82.985878                 5           513


# Analisis estadistico

In [3]:
# Agrupando resultados por threshold
latency_stats = df.groupby('threshold')['latency_ms'].describe()
print(latency_stats.to_string())

            count       mean       std        min        25%        50%        75%         max
threshold                                                                                     
0.1        6038.0  82.215358  4.816453  15.558004  79.757929  81.869602  83.963990  155.927896
0.5        6038.0  81.968431  3.925378  15.640020  79.705000  81.757069  83.916426  128.760099
0.9        6038.0  82.099020  4.363857  15.995979  79.715371  81.827044  83.970010  130.265236


# Histograma por threshold

In [ ]:
# Función personalizada que añade las frecuencias
def histplot_with_freq(x, color=None, **kwargs):
    # Crear el histograma normal
    ax = sns.histplot(x, bins=10, color=color, kde=True, **kwargs)
    
    # Obtener los datos del histograma
    n, bins, patches = ax.hist(x, bins=10, color=color, **kwargs)
    
    # Agregar las frecuencias encima de cada barra
    for count, patch in zip(n, patches):
        if count > 0:  # Solo mostrar texto si hay frecuencia
            ax.text(patch.get_x() + patch.get_width()/2, 
                    patch.get_height(), 
                    f'{int(count)}', 
                    ha='center', 
                    va='bottom',
                    fontsize=9,
                    bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', pad=1))

# Configuración del estilo
plt.figure(figsize=(12, 6))
sns.set_style("whitegrid")

# Crear histogramas separados por threshold con frecuencias
g = sns.FacetGrid(df, col="threshold", col_wrap=3, height=4, sharex=False, sharey=False)
g.map(histplot_with_freq, "latency_ms")

# Añadir títulos y etiquetas
g.set_axis_labels("Latencia (ms)", "Frecuencia")
g.fig.suptitle("Distribución de Latencia por Threshold (con conteos de frecuencia)", y=1.02)

plt.tight_layout()
plt.show()

In [2]:
# Agrupando resultados por threshold
resultados = df.groupby(['video', 'threshold'])['latency_ms'].agg(['max', 'min', 'mean']).reset_index()
resultados.columns = ['video', 'threshold', 'latencia_maxima', 'latencia_minima', 'latencia_promedio']

print(resultados.to_string())

                                    video  threshold  latencia_maxima  latencia_minima  latencia_promedio
0               ejercicio01 - derecha.mp4        0.1       127.338171        76.353788          82.382298
1               ejercicio01 - derecha.mp4        0.5       122.172832        76.689005          82.129775
2               ejercicio01 - derecha.mp4        0.9       126.398087        76.250076          82.293767
3               ejercicio01 - derecho.mp4        0.1        85.664749        77.922106          81.974086
4               ejercicio01 - derecho.mp4        0.5       121.339083        77.877045          83.084393
5               ejercicio01 - derecho.mp4        0.9        86.565971        78.346968          82.182390
6                ejercicio01 - frente.mp4        0.1       124.819040        76.345921          82.483146
7                ejercicio01 - frente.mp4        0.5       125.366211        76.537132          82.078274
8                ejercicio01 - frente.mp4     

In [ ]:
plt.figure(figsize=(3.15, 5))  # Aumentar tamaño para mejor legibilidad

# Boxplot para latencia con estilo mejorado
sns.boxplot(
    data=resultados,
    x='threshold',
    y='latencia_promedio',
    showmeans=True,
    meanprops={'marker':'o', 'markerfacecolor':'white', 'markeredgecolor':'black', 'markersize':10},
    palette='Blues'  # Cambiar paleta de colores
)

# Añadir puntos individuales con estilo mejorado
sns.stripplot(
    data=resultados, 
    x='threshold', 
    y='latencia_promedio',
    color='black', 
    alpha=0.3,  # Reducir transparencia
    jitter=0.2,  # Controlar dispersión
    size=5  # Tamaño de puntos
)

# Personalización del gráfico
plt.title('Distribución de Latencia por Threshold', fontsize=16, pad=20)
plt.xlabel('Threshold', fontsize=14)
plt.ylabel('Latencia (ms)', fontsize=14)
plt.grid(axis='y', linestyle='--', alpha=0.3)

# Añadir anotaciones de estadísticas (opcional - necesitarías calcular las estadísticas primero)
# latencia_stats = resultados.groupby('threshold')['latencia_promedio'].agg(['count', 'mean', 'std'])
# for i, threshold in enumerate(latencia_stats.index):
#     plt.text(
#         i, resultados['latencia_promedio'].max()*1.05,
#         f"n={int(latencia_stats.loc[threshold, 'count'])}\nμ={latencia_stats.loc[threshold, 'mean']:.1f} ± {latencia_stats.loc[threshold, 'std']:.1f}",
#         ha='center',
#         fontsize=12,
#         bbox=dict(facecolor='white', alpha=0.8, edgecolor='gray', boxstyle='round,pad=0.5')
#     )

# Añadir línea horizontal de referencia (opcional)
max_latencia = resultados['latencia_promedio'].max()
plt.axhline(y=max_latencia, color='red', linestyle=':', alpha=0.5, label='Máxima latencia')
plt.legend(loc='upper right')

plt.tight_layout()
plt.show()